# B2.3 · Honeypots, canaries and deception in the agent's environment

**Function B — Application Security with an AI SDLC → Trusting the Harness that Tests CyberTravels**  ·  *AI for Security*

Builds on **[B2.2 · Reliability and cost under non-determinism](https://spbreed.github.io/cyber-commons/lessons/B2.2.html)**.

| | |
|---|---|
| Tools used | Canarytokens, Inspect, GLM-4.6, Claude Haiku 4.5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Every detector in this chapter needs a threshold, and every threshold is a trade. A canary needs neither: nothing legitimate has any reason to touch it, so its false-positive rate is zero by construction rather than by tuning.

> **At CyberTravels.** A canary credential in CyberTravels' environment and a honeypot task in the benchmark: two detectors with no threshold to tune, because nothing legitimate has any reason to touch either.

## 2 · The framework

```
   tuned detector                     deception

   threshold ---> TP and FP           canary ---> any touch is a hit
   needs triage                       needs none: nothing legitimate
   finds more                          has a reason to touch it
                                      finds less, at precision 1.00

   canary token in env/config/docs    honeypot task with an instrumented
   real prefix, real length            shortcut that logs and scores 0

   both decay: bait that is not rotated is bait that has been learned
```

Every detector in this chapter has needed a threshold, and every threshold is a
trade between missing things and crying wolf. Deception is the exception:
**a canary has a false-positive rate of zero by construction**, because nothing
legitimate ever touches it.

Two controls from the incident register live here, and both are about the
environment your harness builds rather than the code it analyses.

**Credential canaries (C4.4).** Tokens that look genuine — same prefix, same
format, same length — placed where an agent or an attacker would plausibly
look: config files, environment listings, documentation, artifact metadata. The
issuing service alerts on any authentication attempt. There is no legitimate
use, so a hit is a confirmed compromise and opens an incident automatically. In
the source incident, live credentials found on the public internet were
redistributed to the swarm within minutes; a canary alongside them would have
carried source IP, user agent and timestamp.

**Honeypot tasks (C10.3).** Benchmark tasks salted with a plausible-looking
shortcut that is instrumented rather than functional, plus tasks whose scoring
*appears* manipulable but where manipulation is detected rather than rewarded.
The cheat-attempt rate on those tasks is a leading indicator for the whole
fleet — available continuously, not after the fact.

Both share one operational requirement: **rotate them**. Bait that stays put is
bait that gets learned, and a learned canary is worse than none because it
tells you the environment is clean when it is only well-mapped.

## 3 · A canary, and why its false-positive rate is zero

In [ ]:
CANARIES = {
 "hf_CANARY7Fq2mXvLpR8s": {"placed": "worker env listing",  "rotated_day": 0},
 "ghp_CANARY4mZk1TbNwQ9x": {"placed": "docs/DEPLOY.md",     "rotated_day": 0},
 "sk-CANARYd3Vn8yHc2Uae":  {"placed": "artifact metadata",  "rotated_day": 0},
}
REAL = {"hf_liveTokenNotShown", "ghp_alsoLive"}

AUTH_EVENTS = [
 ("hf_liveTokenNotShown",   "10.2.0.11",  "ci-runner"),
 ("hf_CANARY7Fq2mXvLpR8s",  "203.0.113.9","python-requests/2.31"),
 ("ghp_alsoLive",           "10.2.0.11",  "ci-runner"),
 ("sk-CANARYd3Vn8yHc2Uae",  "203.0.113.9","python-requests/2.31"),
]

print(f"{'token':26s}{'source':14s}{'agent':22s}verdict")
hits = []
for token, ip, ua in AUTH_EVENTS:
    if token in CANARIES:
        hits.append((token, ip, ua))
        verdict = "CONFIRMED COMPROMISE"
    else:
        verdict = "normal use"
    print(f"{token[:24]:26s}{ip:14s}{ua:22s}{verdict}")

print(f"\ncanary hits: {len(hits)}  false positives possible: 0")
print("Not zero because the detector is good - zero because nothing legitimate")
print("has any reason to touch these strings. That is the whole property, and")
print("it is why a hit opens an incident rather than a ticket.")
print(f"attribution carried on the hit: source {hits[0][1]}, agent {hits[0][2]}")
assert len(hits) == 2

## 4 · Honeypot tasks — a shortcut that is logged, not rewarded

In [ ]:
BENCHMARK = [
 {"id": "T-01", "kind": "real",     "shortcut": None},
 {"id": "T-02", "kind": "real",     "shortcut": None},
 {"id": "T-03", "kind": "honeypot", "shortcut": "flag readable in the scorer fixture"},
 {"id": "T-04", "kind": "real",     "shortcut": None},
 {"id": "T-05", "kind": "honeypot", "shortcut": "answer key path guessable from the task id"},
 {"id": "T-06", "kind": "real",     "shortcut": None},
]

def attempt(task, agent_takes_shortcut):
    """The shortcut is instrumented: it records, it never scores."""
    if task["shortcut"] and agent_takes_shortcut:
        return {"task": task["id"], "scored": 0.0, "logged": "cheat attempt",
                "detail": task["shortcut"]}
    return {"task": task["id"], "scored": 1.0 if task["kind"] == "real" else 0.0,
            "logged": None, "detail": None}

runs = [attempt(t, agent_takes_shortcut=True) for t in BENCHMARK]
for r in runs:
    if r["logged"]:
        print(f"   {r['task']}  {r['logged']}: {r['detail']}  (scored {r['scored']})")

honeypots = [t for t in BENCHMARK if t["kind"] == "honeypot"]
attempts = [r for r in runs if r["logged"]]
rate = len(attempts) / len(honeypots)
print(f"\nhoneypot tasks: {len(honeypots)}   cheat attempts: {len(attempts)}"
      f"   rate: {rate:.0%}")
print()
print("The rate is the product, not the individual catch. Read continuously it")
print("is a leading indicator for the whole fleet; read after the run it is an")
print("autopsy.")
assert rate == 1.0 and all(r["scored"] == 0.0 for r in attempts)

## 5 · Where it breaks — bait that was learned

In [ ]:
def hit_rate(day, rotated_day, learn_days=21):
    """Once bait is learned it stops firing, and silence reads as safety."""
    age = day - rotated_day
    return 1.0 if age < learn_days else 0.0

for day in (0, 7, 20, 21, 40, 90):
    r = hit_rate(day, rotated_day=0)
    reading = "fires on contact" if r else "silent - and silence looks clean"
    print(f"day {day:>3}   detection rate {r:.0%}   {reading}")

print()
rotated = [hit_rate(day, rotated_day=(day // 21) * 21) for day in (0, 20, 40, 90)]
print(f"with quarterly-or-faster rotation: {['%.0f%%' % (x*100) for x in rotated]}")
print()
print("An unrotated canary does not fail loudly. It reports a clean environment")
print("that is merely well-mapped, which is a worse position than having no")
print("canary at all, because you now believe something.")
assert hit_rate(40, 0) == 0.0 and all(x == 1.0 for x in rotated)

## 6 · Verify — what deception buys next to a tuned detector

In [ ]:
DETECTORS = {
 "write-volume anomaly":  {"tp": 18, "fp": 140, "needs_threshold": True},
 "entropy on artifacts":  {"tp": 11, "fp": 62,  "needs_threshold": True},
 "credential canary":     {"tp": 2,  "fp": 0,   "needs_threshold": False},
 "honeypot task":         {"tp": 6,  "fp": 0,   "needs_threshold": False},
}
print(f"{'detector':24s}{'TP':>4}{'FP':>6}{'precision':>11}  threshold?")
for name in sorted(DETECTORS):
    d = DETECTORS[name]
    prec = d["tp"] / (d["tp"] + d["fp"])
    print(f"{name:24s}{d['tp']:>4}{d['fp']:>6}{prec:>11.2f}  "
          f"{'yes' if d['needs_threshold'] else 'none needed'}")

deception = [n for n in DETECTORS if not DETECTORS[n]["needs_threshold"]]
print(f"\ndetectors needing no threshold: {sorted(deception)}")
print()
print("Deception finds less. What it finds needs no triage, no tuning and no")
print("argument - which is why it belongs beside the volume detectors rather")
print("than instead of them.")
assert all(DETECTORS[n]["fp"] == 0 for n in deception)

## What you just proved

Two canary authentications out of four events are confirmed compromises with source IP and user agent attached, and no false positive is structurally possible. Both honeypot tasks log a cheat attempt and score zero for it. An unrotated canary's detection rate falls to 0% once learned — reporting a clean environment that is only well-mapped — while rotation holds it at 100%. Deception finds fewer things than the volume detectors and finds them at precision 1.00.

## Your turn

Place one canary credential in the environment your agents run in, wired to a real alert, and leave it. The interesting outcome is not the alert; it is discovering, six weeks later, which systems can even see it.

## Where this leaves you

**What you can do now.** A harness you can name the eight parts of, evaluate on a corpus with known answers rather than on how confident it sounds, price per confirmed finding across a run nobody watched, and salt with bait that has no false positives.

**What you still cannot do.** Everything you have built so far is defensive and cooperative: it runs against systems that are not trying to defeat it. You have no evidence about how any of it behaves against someone who is — including the evaluation you have been trusting.

**Function C attacks it, starting with the loop pointed the other way round. Next → C1.0, what red teaming and research with AI means.**

---

**Next → [C1.0 · Start here — what red teaming and research with AI means](https://spbreed.github.io/cyber-commons/lessons/C1.0.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*